# Species migration portfolio assignment

## Step 1: Get occurrence data from the Global Biodiversity Information Facility (GBIF)
For your species migration portfolio assignment, you will need to acquire GBIF data for a different species. I recommend choosing a terrestrial species, as this will work best with the ecoregions data used in the assignment.

## Set up
In the imports cell, we’ve included some packages that you will need. Add imports for additional packages that will help you to:
1. Work with reproducible file paths
2. Work with tabular data

In [2]:
import os
import pathlib
import shutil
import time ### time processing steps
import zipfile ### read and extract .zip folders
from getpass import getpass
from glob import glob
import requests

### GBIF packages
import pygbif.occurrences as occ
import pygbif.species as species

### additional packages
import pandas as pd
import geopandas as gpd

### Step 1A: Make a GBIF account
You will need to make a free account with [GBIF](https://www.gbif.org/) to access their data. Keep track of your username, password, and the email address you sign up with, as you'll need all of this later!

You will need a (free) [GBIF account](https://www.gbif.org/) to complete this challenge. Then, run the following code to enter your credentials for the rest of your session.

This code is **interactive**, meaning that it will **ask you for a response**! The prompt can sometimes be
hard to see if you are using VSCode – it appears at the **top** of your editor window.

**Tips:**
1. If you need to save credentials across multiple sessions, you can consider loading them in from a file like a `.env`, but make sure to add it to .gitignore so you don’t commit your credentials to your repository!
2. If you accidentally enter your credentials wrong, you can set `reset = True` instead of `reset = False`.

**Warning**
Your email address **must** match the email you used to sign up for GBIF!

In [18]:
### change this to reset = False after you run this chunk the first time
reset = True

####--------------------------------------####
#### DO NOT MODIFY THE REST OF THIS CODE! ####
####--------------------------------------####
### This code asks for your credentials and saves it for the rest of the session.
### NEVER put your credentials into your code!!!!
### GBIF needs a username, password, and email 

### request and store username
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

### securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password: ')
    
### request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email: ')

### Step 1B: Create a project folder and data directories
The code below will help you get started with making a folder for your data to live in:

In [3]:
### create data directory
data_dir = os.path.join(
    pathlib.Path.home(),
    'earth-analytics',
    'data',
    'sandhill_crane'
)

### make the directory
os.makedirs(data_dir, exist_ok=True)

### define directory for GBIF data
gbif_dir = os.path.join(data_dir, 'gbif')

### make the directory
os.makedirs(gbif_dir, exist_ok=True)

### STEP 1C: Get the taxon key from GBIF

One of the tricky parts about getting occurrence data from GBIF is that species often have multiple names in different contexts. Luckily, GBIF also provides a Name Backbone service that will translate scientific and colloquial names into unique identifiers. GBIF calls these identifiers **taxon keys**:

1. Put the species name into the correct location in the code below.
2. Examine the object you get back from the species query. What part of it do you think might be the taxon key
3. Extract and save the taxon key.

In [4]:
### define the species name
backbone = species.name_backbone(
    scientificName='Antigone canadensis'
)

### check it out
backbone

{'usage': {'key': '9036008',
  'name': 'Antigone canadensis (Linnaeus, 1758)',
  'canonicalName': 'Antigone canadensis',
  'authorship': '(Linnaeus, 1758)',
  'rank': 'SPECIES',
  'code': 'ZOOLOGICAL',
  'status': 'SYNONYM',
  'genericName': 'Antigone',
  'specificEpithet': 'canadensis',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Antigone</i> <i>canadensis</i> (Linnaeus, 1758)'},
 'acceptedUsage': {'key': '2474953',
  'name': 'Grus canadensis (Linnaeus, 1758)',
  'canonicalName': 'Grus canadensis',
  'authorship': '(Linnaeus, 1758)',
  'rank': 'SPECIES',
  'code': 'ZOOLOGICAL',
  'genericName': 'Grus',
  'specificEpithet': 'canadensis',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Grus</i> <i>canadensis</i> (Linnaeus, 1758)'},
 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'},
  {'key': '44', 'name': 'Chordata', 'rank': 'PHYLUM'},
  {'key': '212', 'name': 'Aves', 'rank': 'CLASS'},
  {'key': '1493', 'name': 'Gruiformes', 'rank': 'ORDER'},
  {'key': '9313', 'n

Depending on the output from "backbone," you may need to modify the column in the code below to use 'usageKey' or another column:

In [5]:
### pull out the species key
species_key = backbone['usage']['key']

### check it out
species_key

'9036008'

### Step 1D: Download data from GBIF

Downloading GBIF data is a multi-step process. However, we’ve provided you with a chunk of code that handles the API communications and caches the download. You’ll still need to customize your search.

Submit a request to GBIF using the following steps:
1. Replace `csv_file_pattern` with a string that will match **any** `.csv` file when used in the `.rglob()` method. HINT: The character `*` represents any number of any values except the file separator (e.g. `/` on UNIX systems).
2. Add parameters to the GBIF download function, `occ.download()` to limit your query to:
    - observations of 
    - from 
    - with spatial coordinates.
3. Then, run the download. This can take a few minutes. You can check your downloads by logging on to the [GBIF website](https://www.gbif.org/user/download").

In [17]:
### set file name for download
gbif_pattern = os.path.join(gbif_dir, '*.csv')

### download data only if it doesn't already exist
if not glob(gbif_pattern):

    if 'GBIF_DOWNLOAD_KEY' not in os.environ:

        ### submit request to GBIF
        gbif_query = occ.download([
            f"speciesKey = {species_key}",
            "hasCoordinate = true",
            "year = 2024"
        ])

        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    ### check download status
    download_key = os.environ['GBIF_DOWNLOAD_KEY']
    wait = occ.download_meta(download_key)['status']

    while wait not in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
        time.sleep(5)
        wait = occ.download_meta(download_key)['status']
        print("Download status:", wait)

    if wait != 'SUCCEEDED':
        raise RuntimeError(f"GBIF download status: {wait}")

    ### download the data
    download_info = occ.download_get(
        download_key,
        path=gbif_dir
    )

    ### unzip the data
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path=gbif_dir)

You might notice that the GBIF data filename isn’t very **descriptive**. At this point, you may want to clean up your data directory so that you know what the file is later on!

1. Replace ‘your-gbif-filename’ with a descriptive name.
2. Run the cell.
3. Check your data folder. Is it organized the way you want?

Tip: you can use the `shutil` package to programmatically rename the file.

In [7]:
### find the downloaded CSV file
gbif_pattern = os.path.join(gbif_dir, '*.csv')
gbif_path = glob(gbif_pattern)[0]

### check the file path
gbif_path

'C:\\Users\\Jackson\\earth-analytics\\data\\sandhill_crane\\gbif\\sandhill_crane_gbif.csv'

In [8]:
### give the download a descriptive name
new_gbif_path = pathlib.Path(gbif_dir) / 'sandhill_crane_gbif.csv'

### rename the file
if pathlib.Path(gbif_path) != new_gbif_path:
    shutil.move(gbif_path, new_gbif_path)

### Step 1E: Load the GBIF data into Python

Just like you did when wrangling your data from the data subset, you’ll need to load your GBIF data and convert it to a GeoDataFrame.

In [9]:
print(backbone['usage'])

{'key': '9036008', 'name': 'Antigone canadensis (Linnaeus, 1758)', 'canonicalName': 'Antigone canadensis', 'authorship': '(Linnaeus, 1758)', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'SYNONYM', 'genericName': 'Antigone', 'specificEpithet': 'canadensis', 'type': 'SCIENTIFIC', 'formattedName': '<i>Antigone</i> <i>canadensis</i> (Linnaeus, 1758)'}


In [10]:
### check how many observations GBIF has
results = occ.search(
    taxonKey=species_key,
    hasCoordinate=True,
    year=2024,
    limit=1
)

print("Available observations:", results['count'])

Available observations: 332211


In [11]:
print("File:", new_gbif_path)
print("File size:", os.path.getsize(new_gbif_path))

with open(new_gbif_path, encoding="utf-8-sig") as f:
    for i in range(3):
        print(repr(f.readline()[:300]))
        

File: C:\Users\Jackson\earth-analytics\data\sandhill_crane\gbif\sandhill_crane_gbif.csv
File size: 636
'gbifID\tdatasetKey\toccurrenceID\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecies\tinfraspecificEpithet\ttaxonRank\tscientificName\tverbatimScientificName\tverbatimScientificNameAuthorship\tcountryCode\tlocality\tstateProvince\toccurrenceStatus\tindividualCount\tpublishingOrgKey\tdecimalLatitude\tdecimalLongitude\tc'
''
''


In [23]:
print(backbone['usage'])

{'key': '9036008', 'name': 'Antigone canadensis (Linnaeus, 1758)', 'canonicalName': 'Antigone canadensis', 'authorship': '(Linnaeus, 1758)', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'SYNONYM', 'genericName': 'Antigone', 'specificEpithet': 'canadensis', 'type': 'SCIENTIFIC', 'formattedName': '<i>Antigone</i> <i>canadensis</i> (Linnaeus, 1758)'}


In [26]:
import os
import pygbif.occurrences as occ

### restore your existing GBIF download key
os.environ['GBIF_DOWNLOAD_KEY'] = '0010009-261023074918325'

In [27]:
download_key = os.environ['GBIF_DOWNLOAD_KEY']

download_info = occ.download_meta(download_key)

print("Download status:", download_info.get('status'))
print("Total records:", download_info.get('totalRecords'))
print("Download key:", download_key)

Download status: SUCCEEDED
Total records: 332211
Download key: 0010009-261023074918325


In [14]:
download_info = occ.download_meta(
    "0009979-261023074918325"
)

print(download_info.get("request"))

{'predicate': {'type': 'and', 'predicates': [{'type': 'equals', 'key': 'SPECIES_KEY', 'value': '9036008', 'matchCase': False}, {'type': 'equals', 'key': 'HAS_COORDINATE', 'value': 'true', 'matchCase': False}, {'type': 'equals', 'key': 'YEAR', 'value': '2024', 'matchCase': False}]}, 'sendNotification': True, 'format': 'SIMPLE_CSV', 'type': 'OCCURRENCE', 'verbatimExtensions': [], 'interpretedExtensions': []}


In [29]:
import os
import pathlib
import zipfile
import pygbif.occurrences as occ

### set up your data folder
gbif_dir = os.path.join(
    pathlib.Path.home(),
    'earth-analytics',
    'data',
    'sandhill_crane',
    'gbif'
)

os.makedirs(gbif_dir, exist_ok=True)

### use your successful download
download_key = '0010009-261023074918325'

### download the ZIP file
download_info = occ.download_get(
    download_key,
    path=gbif_dir
)

### extract the data
with zipfile.ZipFile(download_info['path']) as download_zip:
    download_zip.extractall(path=gbif_dir)

print("Download and extraction complete!")

INFO:Download file size: 37008301 bytes
INFO:On disk at C:\Users\Jackson\earth-analytics\data\sandhill_crane\gbif/0010009-261023074918325.zip


Download and extraction complete!


In [30]:
import pandas as pd
from glob import glob

### find CSV files
csv_files = glob(os.path.join(gbif_dir, '*.csv'))

print(csv_files)

['C:\\Users\\Jackson\\earth-analytics\\data\\sandhill_crane\\gbif\\0010009-261023074918325.csv', 'C:\\Users\\Jackson\\earth-analytics\\data\\sandhill_crane\\gbif\\sandhill_crane_gbif.csv']


In [32]:
import geopandas as gpd

### remove records with missing coordinates
gbif_df = gbif_df.dropna(
    subset=['decimalLatitude', 'decimalLongitude']
)

### convert to GeoDataFrame
gbif_gdf = gpd.GeoDataFrame(
    gbif_df,
    geometry=gpd.points_from_xy(
        gbif_df.decimalLongitude,
        gbif_df.decimalLatitude
    ),
    crs="EPSG:4326"
)

### check results
print("Total observations:", len(gbif_gdf))

Total observations: 0


In [34]:
import os
from glob import glob

csv_files = glob(os.path.join(gbif_dir, "*.csv"))

for file in csv_files:
    print("File:", file)
    print("Size:", os.path.getsize(file), "bytes")
    print()

File: C:\Users\Jackson\earth-analytics\data\sandhill_crane\gbif\0010009-261023074918325.csv
Size: 175627959 bytes

File: C:\Users\Jackson\earth-analytics\data\sandhill_crane\gbif\sandhill_crane_gbif.csv
Size: 636 bytes



In [35]:
### select the largest CSV file
gbif_path = max(csv_files, key=os.path.getsize)

print("Selected file:", gbif_path)

### load GBIF observations
gbif_df = pd.read_csv(
    gbif_path,
    sep="\t",
    low_memory=False
)

print("Total observations:", len(gbif_df))

gbif_df.head()

Selected file: C:\Users\Jackson\earth-analytics\data\sandhill_crane\gbif\0010009-261023074918325.csv
Total observations: 332211


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,5279573890,6ff8b3b0-ef0f-4f79-a310-5a5615c6aa0b,1059cbe9-676b-4951-baf9-6f4f1a0816d9,Animalia,Chordata,Aves,Gruiformes,Gruidae,Grus,Grus canadensis,...,NaN,NaN,CC_BY_4_0,NaN,1dd071e3ad977e45482c8f14b9fa82d008b332ab779cdf...,NaN,NaN,2026-10-09T12:07:48.339Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
1,5279690270,6ff8b3b0-ef0f-4f79-a310-5a5615c6aa0b,60264562-2491-409d-9931-4ce18ca29650,Animalia,Chordata,Aves,Gruiformes,Gruidae,Grus,Grus canadensis,...,NaN,NaN,CC_BY_4_0,NaN,36ca66c59f6ef59b585b64923654b3e6812883cf3a6e4e...,NaN,NaN,2026-10-09T12:07:49.875Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
2,5280039181,6ff8b3b0-ef0f-4f79-a310-5a5615c6aa0b,5a83f718-6c28-4b9e-952e-e0469c6466a2,Animalia,Chordata,Aves,Gruiformes,Gruidae,Grus,Grus canadensis,...,NaN,NaN,CC_BY_4_0,NaN,6e576f9908df03e50b390f6da43e1312f7a22dc738e36e...,NaN,NaN,2026-10-09T12:07:54.286Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
3,5280494896,6ff8b3b0-ef0f-4f79-a310-5a5615c6aa0b,1bb4f72e-f037-4dc0-a19f-8cc6a4c1dc3e,Animalia,Chordata,Aves,Gruiformes,Gruidae,Grus,Grus canadensis,...,NaN,NaN,CC_BY_4_0,NaN,a5da96d5e9989cea24dae1720f9afd02c9edb2a021e722...,NaN,NaN,2026-10-09T12:07:59.141Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
4,5277690336,6ff8b3b0-ef0f-4f79-a310-5a5615c6aa0b,af9d42bb-4648-4f9b-84a8-1d96d96f8a38,Animalia,Chordata,Aves,Gruiformes,Gruidae,Grus,Grus canadensis,...,NaN,NaN,CC_BY_4_0,NaN,1dd071e3ad977e45482c8f14b9fa82d008b332ab779cdf...,NaN,NaN,2026-10-09T12:10:29.834Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...


In [37]:
import geopandas as gpd

### remove observations with missing coordinates
gbif_df = gbif_df.dropna(
    subset=['decimalLatitude', 'decimalLongitude']
)

### convert to GeoDataFrame
gbif_gdf = gpd.GeoDataFrame(
    gbif_df,
    geometry=gpd.points_from_xy(
        gbif_df.decimalLongitude,
        gbif_df.decimalLatitude
    ),
    crs="EPSG:4326"
)

### check results
print("Total observations:", len(gbif_gdf))
print("Geographic bounds:", gbif_gdf.total_bounds)

Total observations: 332211
Geographic bounds: [-172.76709    21.366331  177.831069   74.10001 ]


In [38]:
### count observations by month
monthly_counts = (
    gbif_gdf
    .groupby('month')
    .size()
    .reset_index(name='observations')
)

monthly_counts

,month,observations
0,1,21425
1,2,26786
2,3,39451
3,4,43807
4,5,53652
5,6,23003
6,7,15951
7,8,19274
8,9,22346
9,10,22477


In [40]:
import hvplot.pandas

monthly_counts.hvplot.bar(
    x='month',
    y='observations',
    title='Sandhill Crane Observations by Month (2024)',
    xlabel='Month',
    ylabel='Number of Observations',
    color='steelblue'
)

:Bars   [month]   (observations)

In [49]:
import os
import pathlib
import geopandas as gpd
import hvplot.pandas
import cartopy.crs as ccrs
import panel as pn

pn.extension()

ecoreg_path = os.path.join(
    pathlib.Path.home(),
    'earth-analytics',
    'data',
    'veery_migration',
    'ecoregions',
    'ecoregions.shp'
)

ecoreg_gdf = gpd.read_file(ecoreg_path)

ecoreg_gdf.head()

,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
0,1.0,Adelie Land tundra,11.0,Tundra,Antarctica,AN11,1,117,9.749780,0.038948,Half Protected,#63CFAB,#9ED7C2,#257339,CC-BY 4.0,"MULTIPOLYGON (((158.7141 -69.60657, 158.71264 ..."
1,2.0,Admiralty Islands lowland rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Australasia,AU01,2,135,4.800349,0.170599,Nature Could Reach Half Protected,#70A800,#38A700,#7BC141,CC-BY 4.0,"MULTIPOLYGON (((147.28819 -2.57589, 147.2715 -..."
2,3.0,Aegean and Western Turkey sclerophyllous and m...,12.0,"Mediterranean Forests, Woodlands & Scrub",Palearctic,PA12,4,785,162.523044,13.844952,Nature Imperiled,#FF7F7C,#FE0000,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((26.88659 35.32161, 26.88297 35..."
3,4.0,Afghan Mountains semi-desert,13.0,Deserts & Xeric Shrublands,Palearctic,PA13,4,807,15.084037,1.355536,Nature Imperiled,#FA774D,#CC6767,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((65.48655 34.71401, 65.52872 34..."
4,5.0,Ahklun and Kilbuck Upland Tundra,11.0,Tundra,Nearctic,NE11,1,404,22.590087,8.196573,Half Protected,#4C82B6,#9ED7C2,#257339,CC-BY 4.0,"MULTIPOLYGON (((-160.26404 58.64097, -160.2673..."


In [50]:
### match the coordinate reference systems
ecoreg_gdf = ecoreg_gdf.to_crs(gbif_gdf.crs)

### find the ecoregion for each observation
gbif_ecoregion_gdf = (
    ecoreg_gdf
    .sjoin(
        gbif_gdf,
        how='inner',
        predicate='contains'
    )
    [['OBJECTID', 'month']]
    .rename(columns={'OBJECTID': 'ecoregion'})
)

gbif_ecoregion_gdf.head()

,ecoregion,month
4,5.0,8
4,5.0,6
4,5.0,6
4,5.0,9
4,5.0,4


In [53]:
### load ecoregion boundaries
ecoreg_gdf = gpd.read_file(ecoreg_path)

### simplify the geometry
ecoreg_gdf.geometry = ecoreg_gdf.simplify(
    0.05,
    preserve_topology=False
)

### convert to Mercator
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())

### rename the ecoregion column
ecoreg_gdf = ecoreg_gdf.rename(
    columns={'OBJECTID': 'ecoregion'}
)

### set ecoregion as the index
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')

### join occurrence data with ecoregions
occurrence_gdf = ecoreg_gdf.join(occurrence_df)

### check results
occurrence_gdf.head()

ECO_NAME  BIOME_NUM BIOME_NAME  \
ecoregion month                                                           
5.0       4      Ahklun and Kilbuck Upland Tundra       11.0     Tundra   
          6      Ahklun and Kilbuck Upland Tundra       11.0     Tundra   
          8      Ahklun and Kilbuck Upland Tundra       11.0     Tundra   
          9      Ahklun and Kilbuck Upland Tundra       11.0     Tundra   
10.0      4         Alaska-St. Elias Range tundra       11.0     Tundra   

                    REALM ECO_BIOME_  NNH  ECO_ID  SHAPE_LENG  SHAPE_AREA  \
ecoregion month                                                             
5.0       4      Nearctic       NE11    1     404   22.590087    8.196573   
          6      Nearctic       NE11    1     404   22.590087    8.196573   
          8      Nearctic       NE11    1     404   22.590087    8.196573   
          9      Nearctic       NE11    1     404   22.590087    8.196573   
10.0      4      Nearctic       NE11    2     405   98.400727   28.388010   

                                          NNH_NAME    COLOR COLOR_BIO  \
ecoregion month                                                         
5.0       4                         Half Protected  #4C82B6   #9ED7C2   
          6                         Half Protected  #4C82B6   #9ED7C2   
          8                         Half Protected  #4C82B6   #9ED7C2   
          9                         Half Protected  #4C82B6   #9ED7C2   
10.0      4      Nature Could Reach Half Protected  #61D2F2   #9ED7C2   

                COLOR_NNH    LICENSE  \
ecoregion month                        
5.0       4       #257339  CC-BY 4.0   
          6       #257339  CC-BY 4.0   
          8       #257339  CC-BY 4.0   
          9       #257339  CC-BY 4.0   
10.0      4       #7BC141  CC-BY 4.0   

                                                          geometry  \
ecoregion month                                                      
5.0       4      MULTIPOLYGON (((-17912546.29 8050475.591, -179...   
          6      MULTIPOLYGON (((-17912546.29 8050475.591, -179...   
          8      MULTIPOLYGON (((-17912546.29 8050475.591, -179...   
          9      MULTIPOLYGON (((-17912546.29 8050475.591, -179...   
10.0      4      MULTIPOLYGON (((-16886356.561 9047887.14, -168...   

                 occurrences  norm_occurrences  
ecoregion month                                 
5.0       4                4          0.002629  
          6                2          0.002443  
          8                3          0.004195  
          9                2          0.002442  
10.0      4                7          0.001946

In [54]:
### get the plot bounds
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### create the interactive migration map
migration_plot = occurrence_gdf.hvplot(
    c='norm_occurrences',
    groupby='month',

    ### geographic settings
    geo=True,
    crs=ccrs.Mercator(),
    tiles='OSM',

    ### title
    title='Sandhill Crane Migration (2024)',

    ### boundaries
    xlim=(xmin, xmax),
    ylim=(ymin, ymax),

    ### dimensions
    frame_height=600,
    frame_width=900,

    ### widget settings
    widget_location='bottom',
    colorbar=False
)

migration_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'1b0a0795-548e-417c-baec-98e0781086bf': {'version…

AbbreviatedException: ValueError: failed to validate MultiPolygons(id='3000ca7f-9df2-4a57-b584-2330a380a902', ...).fill_color: expected an element of either String, Nullable(Color), Instance(Value), Instance(Field), Instance(Expr), Struct(value=Nullable(Color), transform=Instance(Transform)), Struct(field=String, transform=Instance(Transform)) or Struct(expr=Instance(Expression), transform=Instance(Transform)), got dim('norm_occurrences')

To view the original traceback, catch this exception and call print_traceback() method.

AbbreviatedException: ValueError: failed to validate MultiPolygons(id='3000ca7f-9df2-4a57-b584-2330a380a902', ...).fill_color: expected an element of either String, Nullable(Color), Instance(Value), Instance(Field), Instance(Expr), Struct(value=Nullable(Color), transform=Instance(Transform)), Struct(field=String, transform=Instance(Transform)) or Struct(expr=Instance(Expression), transform=Instance(Transform)), got dim('norm_occurrences')

To view the original traceback, catch this exception and call print_traceback() method.

In [55]:
### create a path for your map
map_path = os.path.join(
    data_dir,
    'sandhill_crane_map.html'
)

### create a version without the basemap
export_plot = occurrence_gdf.hvplot(
    c='norm_occurrences',
    groupby='month',
    geo=True,
    crs=ccrs.Mercator(),
    tiles=None,
    title='Sandhill Crane Migration (2024)',
    xlim=(xmin, xmax),
    ylim=(ymin, ymax),
    frame_height=600,
    frame_width=900,
    widget_location='bottom',
    colorbar=False
)

### export as HTML
pn.panel(export_plot).save(
    map_path,
    embed=True,
    resources='inline'
)

print("Map saved to:", map_path)

Map saved to: C:\Users\Jackson\earth-analytics\data\sandhill_crane\sandhill_crane_map.html


## Wrap up Portfolio Step 1

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [15]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!